# Silver: Job Standardization

This notebook standardizes raw bronze job data into the silver layer with:
- **Canonicalization**: Normalize company names, job titles, locations
- **Field mapping**: Map source-specific fields to standard schema
- **Data type standardization**: Dates, salaries, employment types
- **Record hashing**: Generate fingerprints for change detection
- **Idempotency**: Safe to re-run on same data

## Architecture

**Input**: `bronze.bronze_job_snapshot`  
**Output**: Staged standardized records ready for CDC detection  
**Mode**: Incremental (process new batches only)

In [0]:
# Databricks notebook parameters
dbutils.widgets.text("batch_id", "", "Batch ID to process (leave empty for all unprocessed)")
dbutils.widgets.text("source_filter", "", "Filter by source (comma-separated, empty for all)")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"], "Processing Mode (currently unused)")

# Get parameter values
batch_id = dbutils.widgets.get("batch_id").strip()
source_filter = dbutils.widgets.get("source_filter").strip()
mode = dbutils.widgets.get("mode")

In [0]:
import hashlib
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Configuration
CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Current run metadata
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

In [0]:
# Create metadata table to track processed batches
metadata_table = f"{METADATA_SCHEMA}.bronze_to_silver_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  records_processed INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING,
  PRIMARY KEY (batch_id, source_name)
)
USING DELTA
COMMENT 'Tracks which bronze batches have been processed to silver'
""")

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in bronze
    all_batches_query = f"""
        SELECT DISTINCT batch_id, source_name
        FROM {BRONZE_SCHEMA}.bronze_job_snapshot
    """
    
    # Apply source filter if specified
    if source_filter:
        sources = [f"'{s.strip()}'" for s in source_filter.split(",")]
        all_batches_query += f" WHERE source_name IN ({','.join(sources)})"
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already processed batches
    processed_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unprocessed batches (in bronze but not in metadata)
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unprocessed batches", "records_processed": 0})
    
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
def normalize_company_name(company):
    """Normalize company name for matching"""
    if not company:
        return None
    return company.strip().upper().replace(",", "").replace(".", "").replace("INC", "").replace("LLC", "").replace("LTD", "").replace("GMBH", "").strip()

def normalize_title(title):
    """Normalize job title"""
    if not title:
        return None
    return title.strip().upper()

def normalize_location(location):
    """Normalize location string"""
    if not location:
        return None
    return location.strip().title()

def detect_remote_type(location, remote_flag, description):
    """Detect if job is remote, hybrid, or onsite"""
    if not location and not description:
        return "UNKNOWN"
    
    location_lower = (location or "").lower()
    desc_lower = (description or "").lower()
    
    if remote_flag or "remote" in location_lower or "remote" in desc_lower:
        if "hybrid" in location_lower or "hybrid" in desc_lower:
            return "HYBRID"
        return "REMOTE"
    return "ONSITE"

def generate_record_hash(*fields):
    """Generate deterministic hash for change detection"""
    concatenated = "|".join([str(f) if f is not None else "" for f in fields])
    return hashlib.sha256(concatenated.encode()).hexdigest()

# Register UDFs
normalize_company_udf = F.udf(normalize_company_name, StringType())
normalize_title_udf = F.udf(normalize_title, StringType())
normalize_location_udf = F.udf(normalize_location, StringType())
detect_remote_udf = F.udf(detect_remote_type, StringType())
generate_hash_udf = F.udf(generate_record_hash, StringType())

In [0]:
# Create staging table for CDC detection if it doesn't exist
staging_table = f"{SILVER_SCHEMA}.silver_jobs_staging"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {staging_table} (
  source_name STRING,
  source_job_id STRING,
  source_job_key STRING,
  company_name_raw STRING,
  company_name_norm STRING,
  title_raw STRING,
  title_normalized STRING,
  description_raw STRING,
  location_raw STRING,
  location_norm STRING,
  remote_type STRING,
  source_url STRING,
  posted_at TIMESTAMP,
  last_seen TIMESTAMP,
  batch_id STRING,
  processed_at TIMESTAMP,
  record_hash STRING,
  is_active BOOLEAN,
  soft_delete_flag BOOLEAN,
  soft_delete_reason STRING
)
USING DELTA
COMMENT 'Staging area for standardized jobs before CDC detection'
""")

In [0]:
# Initialize tracking variables
total_records_processed = 0
processed_batch_count = 0
failed_batches = []
all_standardized_records = []

# Define metadata schema (used for recording batch processing status)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("records_processed", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

# Process each batch
for current_batch_id, current_source in batches_to_process:
    try:
        print(f"Processing batch {current_batch_id} ({current_source})...", end=" ")
        
        # Load bronze data for this batch
        query = f"""
            SELECT * FROM {BRONZE_SCHEMA}.bronze_job_snapshot
            WHERE batch_id = '{current_batch_id}'
        """
        
        if current_source:
            query += f" AND source_name = '{current_source}'"
        
        bronze_df = spark.sql(query)
        record_count = bronze_df.count()
        
        if record_count == 0:
            print("No records found")
            continue
        
        # Parse JSON payload to struct
        parsed_df = bronze_df.withColumn("parsed_payload", F.from_json(F.col("payload_json"), "company_name STRING, title STRING, description STRING, location STRING, remote BOOLEAN, url STRING, created_at LONG, candidate_required_location STRING"))
        
        # Apply source-specific field mapping and standardization
        standardized_df = parsed_df.select(
            # Identity
            F.col("source_name"),
            F.col("source_job_id"),
            F.concat_ws("|", F.col("source_name"), F.col("source_job_id")).alias("source_job_key"),
            
            # Company
            F.col("parsed_payload.company_name").alias("company_name_raw"),
            normalize_company_udf(F.col("parsed_payload.company_name")).alias("company_name_norm"),
            
            # Job title
            F.col("parsed_payload.title").alias("title_raw"),
            normalize_title_udf(F.col("parsed_payload.title")).alias("title_normalized"),
            
            # Description
            F.col("parsed_payload.description").alias("description_raw"),
            
            # Location
            F.coalesce(F.col("parsed_payload.location"), F.col("parsed_payload.candidate_required_location")).alias("location_raw"),
            normalize_location_udf(F.coalesce(F.col("parsed_payload.location"), F.col("parsed_payload.candidate_required_location"))).alias("location_norm"),
            
            # Remote type detection
            detect_remote_udf(
                F.coalesce(F.col("parsed_payload.location"), F.col("parsed_payload.candidate_required_location")),
                F.col("parsed_payload.remote"),
                F.col("parsed_payload.description")
            ).alias("remote_type"),
            
            # URL
            F.col("parsed_payload.url").alias("source_url"),
            
            # Timestamps
            F.col("parsed_payload.created_at").cast("timestamp").alias("posted_at"),
            F.col("ingestion_timestamp").alias("last_seen"),
            
            # Batch tracking
            F.col("batch_id"),
            F.lit(run_timestamp).alias("processed_at")
        )
        
        # Generate record hash for change detection
        standardized_df = standardized_df.withColumn(
            "record_hash",
            generate_hash_udf(
                F.col("company_name_norm"),
                F.col("title_normalized"),
                F.col("description_raw"),
                F.col("location_norm"),
                F.col("remote_type")
            )
        )
        
        # Add silver layer metadata
        standardized_df = standardized_df.withColumn(
            "is_active", F.lit(True)
        ).withColumn(
            "soft_delete_flag", F.lit(False)
        ).withColumn(
            "soft_delete_reason", F.lit(None).cast("string")
        )
        
        standardized_count = standardized_df.count()
        
        # Write to staging table
        staging_table = f"{SILVER_SCHEMA}.silver_jobs_staging"
        standardized_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(staging_table)
        
        # Record batch processing in metadata
        metadata_record = spark.createDataFrame([
            (current_batch_id, current_source if current_source else "all", int(standardized_count), run_timestamp_py, run_id, "success")
        ], schema=metadata_schema)
        
        metadata_record.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(metadata_table)
        
        # Update tracking
        total_records_processed += standardized_count
        processed_batch_count += 1
        
        print(f"✓ {standardized_count} records")
        
    except Exception as e:
        print(f"✗ {str(e)}")
        failed_batches.append((current_batch_id, current_source, str(e)))
        
        # Record failure in metadata
        try:
            failure_record = spark.createDataFrame([
                (current_batch_id, current_source if current_source else "all", 0, run_timestamp_py, run_id, f"failed: {str(e)}")
            ], schema=metadata_schema)
            
            failure_record.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(metadata_table)
        except:
            pass
        
        continue

print(f"\nProcessed {processed_batch_count} batches, {total_records_processed} total records")
if failed_batches:
    print(f"Failed: {len(failed_batches)} batches")

In [0]:
# Generate summary statistics for this run
if processed_batch_count > 0:
    summary_df = spark.sql(f"""
    SELECT 
      source_name,
      COUNT(*) as record_count,
      COUNT(DISTINCT company_name_norm) as unique_companies,
      COUNT(DISTINCT title_normalized) as unique_titles,
      COUNT(DISTINCT location_norm) as unique_locations,
      SUM(CASE WHEN remote_type = 'REMOTE' THEN 1 ELSE 0 END) as remote_jobs,
      SUM(CASE WHEN remote_type = 'HYBRID' THEN 1 ELSE 0 END) as hybrid_jobs,
      SUM(CASE WHEN remote_type = 'ONSITE' THEN 1 ELSE 0 END) as onsite_jobs
    FROM {staging_table}
    GROUP BY source_name
    """)
    
    display(summary_df)
    display(spark.table(staging_table).limit(10))

# Show processing metadata
metadata_df = spark.sql(f"""
    SELECT batch_id, source_name, records_processed, processed_at, status
    FROM {metadata_table}
    ORDER BY processed_at DESC
    LIMIT 20
""")
display(metadata_df)

# Return success
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "records_processed": total_records_processed,
    "staging_table": staging_table,
    "metadata_table": metadata_table,
    "next_step": "Run silver_detect_cdc" if processed_batch_count > 0 else "No action needed"
}

dbutils.notebook.exit(json.dumps(result))